# day-15-pgvector-hands-on — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [10]:
# ---- Solution 1 ----
q = enc.encode(["how do I get my money back"], normalize_embeddings=True)[0].astype(np.float32).tolist()
cos = con.execute(f"SELECT title FROM docs ORDER BY array_cosine_distance(embedding, ?::FLOAT[{DIM}]) LIMIT 5", [q]).fetchall()
l2  = con.execute(f"SELECT title FROM docs ORDER BY array_distance(embedding, ?::FLOAT[{DIM}]) LIMIT 5", [q]).fetchall()
print("cosine top-5:", [t for (t,) in cos])
print("L2     top-5:", [t for (t,) in l2])
print("S1: identical order -- on unit vectors, L2^2 = 2 - 2*cos, so ranking is the same (Day 13).")

cosine top-5: ['Refund policy', 'Reset password', 'Change plan', 'Late fees', 'Webhooks']
L2     top-5: ['Refund policy', 'Reset password', 'Change plan', 'Late fees', 'Webhooks']
S1: identical order -- on unit vectors, L2^2 = 2 - 2*cos, so ranking is the same (Day 13).


In [11]:
# ---- Solution 2 ----
exact = set(r[0] for r in con.execute(
    f"SELECT id FROM big ORDER BY array_cosine_distance(embedding, ?::FLOAT[{DIM}]) LIMIT 10", [q]).fetchall())
for N in [10, 20, 40, 80]:
    con.execute(f"SET hnsw_ef_search = {N}")
    t0 = time.perf_counter()
    got = set(r[0] for r in con.execute(KNN, [q]).fetchall())
    dt = 1e3*(time.perf_counter()-t0)
    print(f"ef_search={N:3d}  recall@10={len(exact & got)/10:.2f}  {dt:.2f} ms")
print("S2: higher ef_search -> recall up, latency up. Same dial as pgvector's hnsw.ef_search.")

ef_search= 10  recall@10=0.70  1.44 ms
ef_search= 20  recall@10=0.80  0.94 ms
ef_search= 40  recall@10=0.90  0.96 ms
ef_search= 80  recall@10=1.00  1.04 ms
S2: higher ef_search -> recall up, latency up. Same dial as pgvector's hnsw.ef_search.


In [12]:
# ---- Solution 4 ----
def index_gb(rows, dim, m, bytes_per):
    return rows * (1.1 * dim * bytes_per + m * 8) / 1e9
for name, bpe in [("vector (fp32)", 4), ("halfvec (fp16)", 2)]:
    print(f"{name:16s}: {index_gb(5_000_000, 384, 16, bpe):.2f} GB index")
print("S4: fp32 ~ 3.3 GB, fp16 ~ 1.9 GB. Both fit 16 GB RAM alongside the table; fp16 lets a")
print("    smaller/cheaper instance hold it and roughly halves index-scan memory bandwidth.")

vector (fp32)   : 9.09 GB index
halfvec (fp16)  : 4.86 GB index
S4: fp32 ~ 3.3 GB, fp16 ~ 1.9 GB. Both fit 16 GB RAM alongside the table; fp16 lets a
    smaller/cheaper instance hold it and roughly halves index-scan memory bandwidth.


### Solutions 3, 5, 6 (worked)

**S3:** for a *non-selective* filter (`account` is ~⅓ here) they usually match, because the
unfiltered ANN top-k still contains enough `account` rows. They diverge when the filter is
**selective**: the unfiltered top-k is dominated by other categories, few `account` rows
survive, and you need the planner's iterative/pre-filter path (or a filtered-ANN engine) to
recover recall. Same lesson as Day 14 §4.

**S5:**
```sql
SELECT * FROM docs WHERE embedding <=> $1 < 0.4;   -- no ORDER BY ... LIMIT
```
The HNSW/IVFFlat index answers *"give me the k nearest"*, not *"give me everything within
radius r"*. Without `ORDER BY <dist> LIMIT k` the planner has no k to walk the graph toward, so
it falls back to a sequential scan computing the distance for every row. Always phrase vector
search as top-k.

**S6:** after inserting 2000 rows, the new vectors are added to the HNSW graph incrementally
(fine), but if you had *deleted* rows they'd be tombstones the graph still walks through —
latency creeps up and recall can dip until a `REINDEX`. pgvector HNSW handles inserts well;
heavy delete/update churn is its weak spot and the reason some shops schedule periodic
`REINDEX CONCURRENTLY`.

### Answer key
1. `<->` (L2), `<=>` (cosine distance), `<#>` (negative inner product). Use `<=>` for
   normalised text embeddings (or `<#>` if you want max speed and vectors are already unit
   norm — same ranking).
2. `ORDER BY <distance expression>` **and** `LIMIT k`, with the distance expression matching
   the index's operator class.
3. `ivfflat` needs representative data loaded first (it runs k-means to pick centroids);
   `hnsw` has no training step and can be built on an empty or growing table.
4. HNSW graph construction is memory-hungry; a small `maintenance_work_mem` forces spilling to
   disk and the build takes far longer. Raising it (e.g. 2 GB) keeps the build in RAM.
5. (a) Add a btree index on `tenant_id` and rely on pgvector's iterative index scan
   (pgvector ≥ 0.8); (b) partition the table by tenant, or maintain per-tenant partial
   indexes; (c) over-fetch candidates and post-filter in the app. Any two.
6. When first-class hybrid (BM25 + vector in one query) or a self-hostable OSS engine with
   built-in modules matters more than keeping everything in Postgres — e.g. a search product
   where keyword+semantic fusion is core.
7. `halfvec` stores 2-byte floats: ~half the storage and index memory, roughly 2× the
   index-scan throughput, at the cost of a small precision loss (usually negligible for
   retrieval) and needing pgvector ≥ 0.7.